In [1]:
import sys

import matplotlib
import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__, matplotlib.__version__)


3.13.15 2.3.4 2.3.3 1.7.2 3.10.7


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 3265
data = pd.read_csv("data/spambase.csv")
feature_names = [c for c in data.columns if c != "spam"]
X, y = data[feature_names].to_numpy(), data["spam"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(X.shape, round(100 * y.mean(), 1), X_train.shape, X_test.shape)


(4204, 57) 39.9 (3153, 57) (1051, 57)


# Task 4 - Random Forests & Ensembles

Surname: Bissenbai  
Student ID (last four digits): 3265  
SEED = 3265

I use the same stratified train/test split and five cross-validation folds throughout the notebook.
Spam is class 1, and legitimate mail is class 0. A false positive means that a legitimate email is marked as spam.
All choices are made using the training data. The test set is used for the reports required in the task.


In [3]:
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=SEED)
bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=SEED)
rf = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=SEED, n_jobs=-1)
lr = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), LogisticRegression(max_iter=2000))
bag_lr = BaggingClassifier(lr, n_estimators=50, random_state=SEED)
knn = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), KNeighborsClassifier(n_neighbors=15))
ada = AdaBoostClassifier(n_estimators=400, random_state=SEED)
members = [("lr", lr), ("knn", knn), ("tree", tree)]


Part 1 - Bagging and out-of-bag estimates


In [4]:
def oob_mask(sample_indices, n_samples):
    """Return True for rows that are absent from a bootstrap sample."""
    mask = np.ones(n_samples, dtype=bool)
    indices = np.asarray(sample_indices, dtype=int)
    mask[indices] = False
    return mask


### Checking the function

Repeated indices still mark the same row as in-bag. An empty sample leaves every row out of bag.


In [5]:
assert np.array_equal(oob_mask([0, 2, 2], 5), [False, True, False, True, True])
assert np.array_equal(oob_mask([], 3), [True, True, True])
assert np.array_equal(oob_mask(np.array([3, 0]), 4), [False, True, True, False])

sample = np.array([3, 0, 3])
original_sample = sample.copy()
result = oob_mask(sample_indices=sample, n_samples=4)
assert np.array_equal(sample, original_sample)
assert result.dtype == np.dtype(bool) and result.shape == (4,)
assert oob_mask([], 0).shape == (0,)
print("All oob_mask checks passed.")


All oob_mask checks passed.


### 1. Out-of-bag share

Each tree draws as many rows as the training set contains, with replacement.
Some rows are repeated, so the tree does not see every training row.
For a particular row, the probability of being left out is $(1 - 1/n)^n$.


In [6]:
bag.fit(X_train, y_train)

n_train = len(y_train)
oob_masks = [oob_mask(indices, n_train) for indices in bag.estimators_samples_]
oob_shares = np.array([mask.mean() for mask in oob_masks])
expected_oob = (1 - 1 / n_train) ** n_train

share_table = pd.DataFrame({
    "Observed mean": [oob_shares.mean(), 1 - oob_shares.mean()],
    "Formula": [expected_oob, 1 - expected_oob],
}, index=["Out of bag", "In bag (unique rows)"])
share_table.round(6)


,Observed mean,Formula
Out of bag,0.367796,0.367821
In bag (unique rows),0.632204,0.632179


### 2. Out-of-bag accuracy by hand

For each row, I count votes only from trees that did not train on that row.
The larger count gives the predicted class. With argmax, a tie gives class 0.


In [7]:
oob_votes = np.zeros((n_train, 2), dtype=int)

for estimator, mask in zip(bag.estimators_, oob_masks):
    row_indices = np.flatnonzero(mask)
    predictions = estimator.predict(X_train[mask]).astype(int)
    oob_votes[row_indices, predictions] += 1

print("Rows with no OOB votes:", np.sum(oob_votes.sum(axis=1) == 0))
assert np.all(oob_votes.sum(axis=1) > 0)

oob_predictions = np.argmax(oob_votes, axis=1)
manual_oob_accuracy = np.mean(oob_predictions == y_train)
print(f"Manual OOB accuracy:       {manual_oob_accuracy:.6f}")
print(f"BaggingClassifier OOB:     {bag.oob_score_:.6f}")
print(f"Absolute difference:       {abs(manual_oob_accuracy - bag.oob_score_):.12f}")
assert np.isclose(manual_oob_accuracy, bag.oob_score_)


Rows with no OOB votes: 0
Manual OOB accuracy:       0.933080
BaggingClassifier OOB:     0.933080
Absolute difference:       0.000000000000


### 3. Five models on the same folds

I compare accuracy on the five training folds. cross_validate makes a fresh copy of the model for each fold.
The transformations for logistic regression stay inside its pipeline, so the scaler only sees the training rows of each fold.
The standard deviation uses ddof=0, as required.


In [8]:
from sklearn.model_selection import cross_validate

models_part1 = {"tree": tree, "bag": bag, "rf": rf, "lr": lr, "bag_lr": bag_lr}
cv_scores = {}
cv_rows = []

for name, model in models_part1.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring="accuracy")["test_score"]
    cv_scores[name] = scores
    cv_rows.append([name, *scores, scores.mean(), scores.std()])

score_columns = ["Model", "Fold 1", "Fold 2", "Fold 3", "Fold 4", "Fold 5", "Mean", "Std"]
cv_table_part1 = pd.DataFrame(cv_rows, columns=score_columns).set_index("Model")
cv_table_part1.round(6)


,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean,Std
Model,,,,,,,
tree,0.889065,0.885895,0.881141,0.906349,0.907937,0.894077,0.010974
bag,0.923930,0.923930,0.923930,0.942857,0.949206,0.932771,0.011012
rf,0.935024,0.942948,0.939778,0.944444,0.955556,0.943550,0.006815
lr,0.928685,0.936609,0.920761,0.928571,0.947619,0.932449,0.009091
bag_lr,0.930269,0.938193,0.920761,0.928571,0.947619,0.933083,0.009138


In [9]:
paired_differences_part1 = {}
difference_rows = []

for first, second in [("bag", "tree"), ("rf", "tree"), ("bag_lr", "lr")]:
    name = f"{first} - {second}"
    differences = cv_scores[first] - cv_scores[second]
    paired_differences_part1[name] = differences
    difference_rows.append([
        name, *differences, differences.mean(), differences.std(),
        abs(differences.mean()) <= differences.std(),
    ])

difference_columns = ["Comparison", "Fold 1", "Fold 2", "Fold 3", "Fold 4", "Fold 5",
                      "Mean", "Std", "Inside fold spread"]
paired_table_part1 = pd.DataFrame(difference_rows, columns=difference_columns).set_index("Comparison")
paired_table_part1.round(6)


,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean,Std,Inside fold spread
Comparison,,,,,,,,
bag - tree,0.034865,0.038035,0.042789,0.036508,0.041270,0.038693,0.002942,False
rf - tree,0.045959,0.057052,0.058637,0.038095,0.047619,0.049472,0.007572,False
bag_lr - lr,0.001585,0.001585,0.000000,0.000000,0.000000,0.000634,0.000776,True


### 4. One forest, three estimates

I fit the fixed random forest on the full training set and keep it for the next part.
The test standard error is $\sqrt{p(1-p)/n_{test}}$, where $p$ is its test accuracy.


In [10]:
rf.fit(X_train, y_train)

rf_cv_mean = cv_scores["rf"].mean()
rf_cv_std = cv_scores["rf"].std()
rf_test_accuracy = rf.score(X_test, y_test)
rf_test_se = np.sqrt(rf_test_accuracy * (1 - rf_test_accuracy) / len(y_test))

forest_estimates = pd.DataFrame({
    "Estimate": ["OOB accuracy", "CV mean accuracy", "Test accuracy"],
    "Accuracy": [rf.oob_score_, rf_cv_mean, rf_test_accuracy],
}).set_index("Estimate")
display(forest_estimates.round(6))
print(f"CV standard deviation: {rf_cv_std:.6f}")
print(f"Test standard error:   {rf_test_se:.6f}")
print(f"OOB-CV absolute gap:   {abs(rf.oob_score_ - rf_cv_mean):.6f}")
print("OOB and CV agree by the task rule:", abs(rf.oob_score_ - rf_cv_mean) <= rf_cv_std)
print(f"OOB-test absolute gap: {abs(rf.oob_score_ - rf_test_accuracy):.6f}")
print(f"CV-test absolute gap:  {abs(rf_cv_mean - rf_test_accuracy):.6f}")


,Accuracy
Estimate,
OOB accuracy,0.944497
CV mean accuracy,0.943550
Test accuracy,0.957184


CV standard deviation: 0.006815
Test standard error:   0.006245
OOB-CV absolute gap:   0.000947
OOB and CV agree by the task rule: True
OOB-test absolute gap: 0.012686
CV-test absolute gap:  0.013634


### What the results show

**Bootstrap and OOB.** The mean OOB share is 0.367796, close to the formula value of 0.367821. This means that each tree leaves out about 36.8% of the training rows. The manual OOB accuracy matches bag.oob_score_, and every training row has at least one OOB vote.

**Bagging trees.** Mean CV accuracy increases from 0.894077 for one tree to 0.932771 for bagging. The paired improvement is 0.038693 with a standard deviation of 0.002942. It is outside the fold spread, so bagging helps the tree here. A single unrestricted tree can change a lot when its training sample changes; combining bootstrap trees reduces this variance.

**Random forest versus one tree.** The forest's mean CV accuracy is 0.943550. Its paired improvement over the tree is 0.049472 with a standard deviation of 0.007572, also outside the fold spread. In addition to bootstrap sampling, the forest considers a random subset of features at each split, which makes the trees less similar.

**Bagging logistic regression.** The mean accuracy changes from 0.932449 to 0.933083. The paired improvement is only 0.000634, smaller than its standard deviation of 0.000776. This is inside the fold spread, so these results do not support choosing bagged logistic regression over the single model. Logistic regression is usually more stable than an unrestricted tree, so averaging bootstrap fits has less variance to reduce.

**The forest's three estimates.** OOB accuracy is 0.944497 and mean CV accuracy is 0.943550. Their gap of 0.000947 is below the CV standard deviation of 0.006815, so OOB is a fair stand-in for CV by the task's rule. Test accuracy is 0.957184 with a standard error of 0.006245. The test result is higher than OOB by 0.012686 and higher than CV by 0.013634; both gaps exceed one test standard error. OOB and CV therefore agree with each other, but are somewhat lower than this test result. The test standard error describes uncertainty in the test accuracy alone; it is not a formal test of the difference between these estimates.
